# 40 — The greedy λ dial for composition (λ=0 vs λ=0.4, n=2..4)

Notebook 31 found that on greedy *single-concept* steering the fluency
weight λ=0.4 improves steering AND fluency simultaneously (Δ 0.41→0.81,
ppl 7.5→2.3) — the fluency term keeps greedy out of degenerate paths.
The entire greedy-tier scaling suite (33, 39), however, runs λ=0 only;
the composition λ ablation (E1b) was beam-tier. This notebook fills the
gap: **greedy blend/vote/schedule at λ=0.4 across the full nested groups
(10 pairs + 30 triples + 30 quads)**, compared against the existing λ=0
rows from notebooks 33 and 39.

Questions: does the greedy-singles λ effect transfer to composition? Does
it interact with n? Are the greedy-tier scaling curves understated?

Implementation note: the canonical single-guide wrapper has no fluency
dial (golden-pinned), so the blend arm routes through the unified loop
with a single composite guide — the same sanctioned path notebook 31
used.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from frames.representations.schedulers import RoundRobin

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
LAM = 0.4

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

GROUPS33 = json.loads(Path("resources/33_scaling_groups_bf16.json").read_text())
QUADS = json.loads(Path("resources/39_quads.json").read_text())["quads"]
GROUPS = [*GROUPS33["pairs"], *GROUPS33["triples"], *QUADS]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/40_greedy_lambda.jsonl")
LOG_PATH.unlink(missing_ok=True)
print(f"{len(GROUPS)} groups (pairs + triples + quads)")

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
prompts = [chat(u, a) for u, a in QUESTIONS]
harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)

used = sorted(
    {s for g in GROUPS for s in (g["a"], g["b"], g.get("c"), g.get("d")) if s}
)
concept = {}
for name in used:
    c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    concept[name] = Concept(synset=c.synset, tensor=c.tensor.to(fur.model.device))
print(f"{len(concept)} concepts built")

for group in GROUPS:
    synsets = [s for s in (group["a"], group["b"], group.get("c"), group.get("d")) if s]
    cs = [concept[s] for s in synsets]
    blend_guide = Concept.average(cs)
    methods = {
        # canonical wrapper has no fluency dial; unified loop w/ single guide
        "blend": lambda: fur.generate_with_topk_multi_guide(
            prompts, guides=[blend_guide], k=K, steps=STEPS, fluency_weight=LAM
        ),
        "vote": lambda: fur.generate_with_topk_multi_guide(
            prompts,
            guides=cs,
            k=K,
            steps=STEPS,
            normalize="zscore",
            fluency_weight=LAM,
        ),
        "schedule": lambda: fur.generate_with_topk_multi_guide(
            prompts,
            guides=cs,
            k=K,
            steps=STEPS,
            scorer=RoundRobin(),
            fluency_weight=LAM,
        ),
    }
    for method, generate in methods.items():
        torch.cuda.empty_cache()
        texts, probe = generate()
        harness.evaluate(
            prompts,
            texts,
            synsets,
            config={
                "model": MODEL_KEY,
                "method": method,
                "n": len(synsets),
                "lam": LAM,
                **group,
            },
            probe=probe,
            baseline_texts=baseline_texts,
            concepts=dict(zip(synsets, cs)),
            angles=True,
        )
    print(f"n={len(synsets)} {' + '.join(s.split('.')[0] for s in synsets)}: done")

In [ ]:
def rows_from(path, methods):
    rows = []
    for line in Path(path).read_text().strip().split("\n"):
        r = json.loads(line)
        c = r["config"]
        if c["method"] not in methods:
            continue
        synsets = [s for s in (c["a"], c["b"], c.get("c"), c.get("d")) if s]
        deltas = [r["expression"][s]["delta"] for s in synsets]
        cosines = [r["angles"][s]["delta"]["token_cos"] for s in synsets]
        rows.append(
            {
                "method": c["method"],
                "n": c["n"],
                "lam": c.get("lam", 0.0),
                "joint": min(deltas),
                "joint_cos": min(cosines),
                "all_positive_cos": min(cosines) > 0,
                "ppl_ratio": r["ppl_ratio"],
            }
        )
    return pd.DataFrame(rows)


plain = {"blend", "vote", "schedule"}
results = pd.concat(
    [
        rows_from(LOG_PATH, plain),
        rows_from("resources/33_scaling3_bf16.jsonl", plain),
        rows_from("resources/39_scaling4.jsonl", plain),
    ],
    ignore_index=True,
)
print(f"gate: {results.groupby(['lam', 'n']).size().to_dict()} records")

summary = (
    results.groupby(["method", "n", "lam"])
    .agg(
        joint=("joint", "mean"),
        joint_cos=("joint_cos", "mean"),
        all_positive_cos=("all_positive_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n_runs=("joint", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/40_greedy_lambda_summary.csv", index=False)
summary.round(4)

In [ ]:
methods = ["blend", "vote", "schedule"]
colors = {"blend": "tab:blue", "vote": "tab:orange", "schedule": "tab:green"}
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
for ax, (metric, label) in zip(
    axes,
    (
        ("joint_cos", "mean joint token-cosine gain"),
        ("all_positive_cos", "share ALL concepts improved (cosine)"),
        ("ppl_ratio", "mean perplexity ratio"),
    ),
):
    for method in methods:
        for lam, ls in ((0.0, "-"), (0.4, "--")):
            sub = summary[
                (summary["method"] == method) & (summary["lam"] == lam)
            ].sort_values("n")
            ax.plot(
                sub["n"],
                sub[metric],
                marker="o",
                ls=ls,
                color=colors[method],
                label=f"{method} lam={lam}",
            )
    ax.set_xticks([2, 3, 4])
    ax.set_xlabel("concurrent concepts n")
    ax.set_ylabel(label)
axes[0].axhline(0, color="gray", lw=0.8)
axes[1].axhline(0.5, color="gray", ls=":", lw=0.8)
axes[2].axhline(2.5, color="gray", ls=":", lw=0.8)
axes[0].legend(fontsize=7)
plt.suptitle("Greedy composition: lambda=0 (solid) vs lambda=0.4 (dashed)")
plt.tight_layout()
plt.savefig("resources/40_greedy_lambda.png", dpi=150)
plt.show()

print("lambda effect (0 -> 0.4) per method and n:")
for method in methods:
    for n in (2, 3, 4):
        s0 = summary[
            (summary["method"] == method)
            & (summary["n"] == n)
            & (summary["lam"] == 0.0)
        ]
        s4 = summary[
            (summary["method"] == method)
            & (summary["n"] == n)
            & (summary["lam"] == 0.4)
        ]
        print(
            f"  {method:9s} n={n}: joint_cos {s0['joint_cos'].iloc[0]:.4f} -> "
            f"{s4['joint_cos'].iloc[0]:.4f}   ppl {s0['ppl_ratio'].iloc[0]:.1f} -> "
            f"{s4['ppl_ratio'].iloc[0]:.1f}"
        )